In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "dbw_retail_lakehouse_dev_eas_001"

BRONZE = f"{CATALOG}.bronze"
SILVER = f"{CATALOG}.silver"

In [0]:
df_customers = spark.table(f"{BRONZE}.customers")

display(df_customers)

In [0]:
df_customers_clean = (
    df_customers
    .withColumn("customer_id", F.trim("customer_id"))
    .withColumn("first_name", F.initcap(F.trim("first_name")))
    .withColumn("last_name", F.initcap(F.trim("last_name")))
    .withColumn("email", F.lower(F.trim("email")))
    .withColumn("country", F.upper(F.trim("country")))
)

In [0]:
df_customers_valid = (
    df_customers_clean
    .filter(
        F.col("customer_id").isNotNull() &
        (F.col("customer_id") != "") &
        F.col("email").isNotNull() &
        (F.col("email") != "") &
        F.col("created_at").isNotNull()
    )
)

In [0]:
customer_window = (
    Window
    .partitionBy("customer_id")
    .orderBy(F.col("_ingested_at").desc())
)

df_customers_silver = (
    df_customers_valid
    .withColumn(
        "_row_number",
        F.row_number().over(customer_window)
    )
    .filter(F.col("_row_number") == 1)
    .drop("_row_number")
)

In [0]:
(
    df_customers_silver
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{SILVER}.customers")
)

In [0]:
display(spark.table(f"{SILVER}.customers"))

print(
    "rows:",
    spark.table(f"{SILVER}.customers").count()
)

In [0]:
df_products = spark.table(f"{BRONZE}.products")

In [0]:
df_products_clean = (
    df_products
    .withColumn("product_id", F.upper(F.trim("product_id")))
    .withColumn("product_name", F.trim("product_name"))
    .withColumn("category", F.initcap(F.trim("category")))
)

In [0]:
df_products_valid = (
    df_products_clean
    .filter(
        F.col("product_id").isNotNull() &
        (F.col("product_id") != "") &
        F.col("product_name").isNotNull() &
        (F.col("product_name") != "") &
        F.col("unit_price").isNotNull() &
        (F.col("unit_price") >= 0)
    )
)

In [0]:
product_window = (
    Window
    .partitionBy("product_id")
    .orderBy(F.col("_ingested_at").desc())
)

df_products_silver = (
    df_products_valid
    .withColumn(
        "_row_number",
        F.row_number().over(product_window)
    )
    .filter(F.col("_row_number") == 1)
    .drop("_row_number")
)

In [0]:
(
    df_products_silver
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{SILVER}.products")
)

In [0]:
df_orders = spark.table(f"{BRONZE}.orders")

In [0]:
df_orders_clean = (
    df_orders
    .withColumn("order_id", F.upper(F.trim("order_id")))
    .withColumn("customer_id", F.upper(F.trim("customer_id")))
    .withColumn("product_id", F.upper(F.trim("product_id")))
    .withColumn("status", F.lower(F.trim("status")))
)

In [0]:
df_orders_valid = (
    df_orders_clean
    .filter(
        F.col("order_id").isNotNull() &
        F.col("customer_id").isNotNull() &
        F.col("product_id").isNotNull() &
        F.col("order_timestamp").isNotNull() &
        (F.col("quantity") > 0) &
        (F.col("unit_price") >= 0) &
        F.col("status").isin(
            "pending",
            "completed",
            "cancelled"
        )
    )
)

In [0]:
valid_customers = (
    spark.table(f"{SILVER}.customers")
    .select("customer_id")
)

valid_products = (
    spark.table(f"{SILVER}.products")
    .select("product_id")
)

In [0]:
df_orders_valid = (
    df_orders_valid
    .join(
        valid_customers,
        on="customer_id",
        how="inner"
    )
)

In [0]:
df_orders_valid = (
    df_orders_valid
    .join(
        valid_products,
        on="product_id",
        how="inner"
    )
)

In [0]:
order_window = (
    Window
    .partitionBy(
        "order_id",
        "product_id"
    )
    .orderBy(F.col("_ingested_at").desc())
)

df_orders_silver = (
    df_orders_valid
    .withColumn(
        "_row_number",
        F.row_number().over(order_window)
    )
    .filter(F.col("_row_number") == 1)
    .drop("_row_number")
)

In [0]:
df_orders_silver = (
    df_orders_silver
    .withColumn(
        "line_amount",
        F.col("quantity") * F.col("unit_price")
    )
)

In [0]:
(
    df_orders_silver
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{SILVER}.orders")
)

In [0]:
display(
    spark.table(f"{SILVER}.orders")
)

In [0]:
df_inventory = spark.table(f"{BRONZE}.inventory")

In [0]:
df_inventory_clean = (
    df_inventory
    .withColumn(
        "product_name",
        F.trim("product_name")
    )
)

In [0]:
df_inventory_valid = (
    df_inventory_clean
    .filter(
        F.col("product_id").isNotNull() &
        F.col("product_name").isNotNull() &
        (F.col("product_name") != "") &
        F.col("quantity_on_hand").isNotNull() &
        (F.col("quantity_on_hand") >= 0)
    )
)

In [0]:
inventory_window = (
    Window
    .partitionBy("product_id")
    .orderBy(F.col("_ingested_at").desc())
)

df_inventory_silver = (
    df_inventory_valid
    .withColumn(
        "_row_number",
        F.row_number().over(inventory_window)
    )
    .filter(F.col("_row_number") == 1)
    .drop(
        "_row_number",
        "_api_total",
        "_api_skip",
        "_api_limit"
    )
)

In [0]:
(
    df_inventory_silver
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{SILVER}.inventory")
)

In [0]:
tables = [
    "customers",
    "products",
    "orders",
    "inventory"
]

for table in tables:
    bronze_count = spark.table(
        f"{BRONZE}.{table}"
    ).count()

    silver_count = spark.table(
        f"{SILVER}.{table}"
    ).count()

    print(
        f"{table:10} "
        f"bronze={bronze_count:3} "
        f"silver={silver_count:3} "
        f"rejected={bronze_count - silver_count:3}"
    )

In [0]:
%sql

SHOW TABLES
IN dbw_retail_lakehouse_dev_eas_001.silver;

In [0]:
%sql

SELECT 'customers' AS table_name, COUNT(*) AS row_count
FROM dbw_retail_lakehouse_dev_eas_001.silver.customers

UNION ALL

SELECT 'products', COUNT(*)
FROM dbw_retail_lakehouse_dev_eas_001.silver.products

UNION ALL

SELECT 'orders', COUNT(*)
FROM dbw_retail_lakehouse_dev_eas_001.silver.orders

UNION ALL

SELECT 'inventory', COUNT(*)
FROM dbw_retail_lakehouse_dev_eas_001.silver.inventory;